<a href="https://colab.research.google.com/github/alizafarbaig/Truth-Table-Generator/blob/main/Truth_Table_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# 🧠 Interactive Truth Table Generator for Google Colab
#    Supports: AND (&, and, ∧), OR (|, or, ∨), NOT (!, not, ¬),
#              XOR (^, xor, ⊕), IMPLIES (->, implies, →),
#              EQUIV (<->, ==, iff, ↔)  and parentheses.
#    Variables: any sequence of letters (e.g., A, B, Rain, Snow).

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import re, itertools
from typing import List, Dict, Tuple

# ----------------------------------------------------------------------
#  Tokenizer, Parser & AST (same logic as the CLI version)
# ----------------------------------------------------------------------
TOKEN_SPEC = [
    ('NOT',    r'!|not|¬'),
    ('AND',    r'&|and|∧'),
    ('OR',     r'\|or|∨'),
    ('XOR',    r'\^|xor|⊕'),
    ('IMPLIES', r'->|implies|→'),
    ('EQUIV',   r'<->|==|iff|↔'),
    ('LPAREN', r'\('),
    ('RPAREN', r'\)'),
    ('VAR',    r'[A-Za-z]+'),    # multi‑letter variables allowed
    ('SKIP',   r'[ \t]+'),
    ('MISMATCH', r'.'),
]
TOKEN_RE = '|'.join(f'(?P<{name}>{pattern})' for name, pattern in TOKEN_SPEC)

def tokenize(expr: str) -> List[Tuple[str, str]]:
    tokens = []
    for match in re.finditer(TOKEN_RE, expr):
        kind = match.lastgroup
        value = match.group()
        if kind == 'SKIP':
            continue
        elif kind == 'MISMATCH':
            raise SyntaxError(f"Unexpected character: {value}")
        elif kind == 'VAR':
            tokens.append(('VAR', value.upper()))
        else:
            tokens.append((kind, value))
    tokens.append(('EOF', ''))
    return tokens

class ASTNode:
    def eval(self, env: Dict[str, bool]) -> bool: raise NotImplementedError

class VarNode(ASTNode):
    def __init__(self, name): self.name = name
    def eval(self, env): return env[self.name]

class UnaryNode(ASTNode):
    def __init__(self, op, child): self.op = op; self.child = child
    def eval(self, env):
        if self.op == 'NOT': return not self.child.eval(env)
        raise ValueError(f"Unknown unary op {self.op}")

class BinaryNode(ASTNode):
    def __init__(self, op, left, right): self.op = op; self.left = left; self.right = right
    def eval(self, env):
        l = self.left.eval(env); r = self.right.eval(env)
        if self.op == 'AND': return l and r
        if self.op == 'OR':  return l or r
        if self.op == 'XOR': return l ^ r
        if self.op == 'IMPLIES': return (not l) or r
        if self.op == 'EQUIV': return l == r
        raise ValueError(f"Unknown binary op {self.op}")

PRECEDENCE = {'EQUIV':1, 'IMPLIES':2, 'OR':3, 'XOR':4, 'AND':5}
RIGHT_ASSOC = {'IMPLIES'}

class Parser:
    def __init__(self, tokens): self.tokens = tokens; self.pos = 0; self.current = tokens[0]
    def advance(self): self.pos += 1; self.current = self.tokens[self.pos]
    def expect(self, kind):
        if self.current[0] != kind: raise SyntaxError(f"Expected {kind}, got {self.current}")
        val = self.current[1]; self.advance(); return val

    def parse(self):
        node = self._parse_expr(0)
        if self.current[0] != 'EOF': raise SyntaxError("Unexpected token")
        return node

    def _parse_expr(self, min_prec):
        token = self.current
        if token[0] == 'NOT':
            self.advance()
            left = UnaryNode('NOT', self._parse_expr(PRECEDENCE['AND']+1))
        elif token[0] == 'VAR':
            self.advance(); left = VarNode(token[1])
        elif token[0] == 'LPAREN':
            self.advance(); left = self._parse_expr(0); self.expect('RPAREN')
        else:
            raise SyntaxError(f"Unexpected token {token}")

        while self.current[0] in PRECEDENCE:
            op = self.current[0]; prec = PRECEDENCE[op]
            if prec < min_prec: break
            next_min = prec + 1 if op in RIGHT_ASSOC else prec
            self.advance()
            right = self._parse_expr(next_min)
            left = BinaryNode(op, left, right)
        return left

# ----------------------------------------------------------------------
#  Variable extraction & truth table generation
# ----------------------------------------------------------------------
def collect_vars(node) -> List[str]:
    seen = set(); order = []
    def walk(n):
        if isinstance(n, VarNode):
            if n.name not in seen:
                seen.add(n.name); order.append(n.name)
        elif isinstance(n, UnaryNode): walk(n.child)
        elif isinstance(n, BinaryNode): walk(n.left); walk(n.right)
    walk(node)
    return order

def build_html_table(expression: str, ast: ASTNode, variables: List[str]) -> str:
    """Generate an HTML table with all variable assignments and the result."""
    n = len(variables)
    if n == 0:
        # expression with no variables (should be rare after parsing)
        result = ast.eval({})
        return f"<p>Constant expression: <b>{int(result)}</b></p>"
    header = variables + [expression]
    rows_html = []
    for values in itertools.product([False, True], repeat=n):
        env = dict(zip(variables, values))
        res = ast.eval(env)
        row_cells = ''.join(f'<td>{"1" if v else "0"}</td>' for v in values)
        row_cells += f'<td><b>{"1" if res else "0"}</b></td>'
        rows_html.append(f'<tr>{row_cells}</tr>')
    header_cells = ''.join(f'<th>{h}</th>' for h in header)
    html = f'''
    <style>
        .truth-table {{ border-collapse: collapse; margin: 10px 0; font-family: monospace; }}
        .truth-table th, .truth-table td {{ border: 1px solid #ccc; padding: 6px 12px; text-align: center; }}
        .truth-table th {{ background: #f0f0f0; }}
        .truth-table tr:nth-child(even) {{ background: #fafafa; }}
    </style>
    <table class="truth-table">
        <thead><tr>{header_cells}</tr></thead>
        <tbody>{''.join(rows_html)}</tbody>
    </table>'''
    return html

# ----------------------------------------------------------------------
#  GUI with ipywidgets
# ----------------------------------------------------------------------
def on_generate(_):
    """Callback for button click or Enter key."""
    out.clear_output()
    expr = text_input.value.strip()
    if not expr:
        with out:
            display(HTML("<p style='color:red'>Please enter a logical expression.</p>"))
        return
    try:
        tokens = tokenize(expr)
        parser = Parser(tokens)
        ast = parser.parse()
        vars_list = collect_vars(ast)
        html = build_html_table(expr, ast, vars_list)
        with out:
            display(HTML(html))
    except Exception as e:
        with out:
            display(HTML(f"<p style='color:red'><b>Error:</b> {e}</p>"))

# ---- Build the UI ----
text_input = widgets.Text(
    value='',
    placeholder='e.g. (A and B) -> C',
    description='Expression:',
    layout=widgets.Layout(width='80%')
)
text_input.on_submit(on_generate)   # press Enter to generate

button = widgets.Button(
    description='Generate Truth Table',
    button_style='primary',
    icon='table'
)
button.on_click(on_generate)

# Output area for the table
out = widgets.Output()

# Display everything
display(
    widgets.HTML("<h3>Truth Table Generator</h3>"
                 "<p>Use <b>A-Z</b> (or longer names) and operators: "
                 "<code>and</code>, <code>or</code>, <code>not</code>, <code>xor</code>, "
                 "<code>-></code> (implies), <code><-></code> (equivalence).</p>"),
    text_input,
    button,
    out
)

HTML(value='<h3>Truth Table Generator</h3><p>Use <b>A-Z</b> (or longer names) and operators: <code>and</code>,…

Text(value='', description='Expression:', layout=Layout(width='80%'), placeholder='e.g. (A and B) -> C')

Button(button_style='primary', description='Generate Truth Table', icon='table', style=ButtonStyle())

Output()